# ML-Hub API — Pipeline Examples

This notebook demonstrates how to run ML pipelines via the REST API.

**Prerequisites:**
- Backend running at `http://localhost:8000`
- Valid HuggingFace token configured (for model/dataset access)

In [ ]:
import requests

BASE_URL = "http://localhost:8000/api"

## 1. Fine-tune a Causal LM

Train a LoRA adapter on a causal language model using your dataset.

This pipeline:
- Loads a base model from HuggingFace (e.g., `mistralai/Mistral-7B-v0.1`)
- Fine-tunes it on your dataset with LoRA + 4-bit quantization
- Saves and pushes the merged model to HuggingFace Hub
- Tracks everything in MLflow

In [ ]:
# Fine-tune a causal language model
response = requests.post(
    f"{BASE_URL}/pipelines/finetune-causal/run",
    json={
        # Required
        "model_name": "mistralai/Mistral-7B-v0.1",
        "dataset_name": "dataesr/my-training-dataset",
        "dataset_split": "train",

        # Custom envs
        "envs": [
            # {"name": "HF_TOKEN", "value": "hf_..."}, 
            {"name": "HF_PUSH_REPO", "value": "dataesr/my-finetuned-model"}, # needed to push the model to huggingface
        ],
        
        # Prompt config (optional - stored in configs/)
        "prompts_config": "my-prompt-template",
        
        # LoRA hyperparameters (optional)
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        
        # Training hyperparameters (optional)
        "epochs": 3,
        "batch_size": 1,
        "learning_rate": 2e-5,
        "max_seq_length": 8192,
    }
)

print(response.status_code)
print(response.json())

## 2. Run Inference on a Dataset

Generate completions for an entire dataset using a fine-tuned (or base) model.

This pipeline:
- Loads your model with vLLM for fast batched inference
- Runs inference on all prompts in the dataset
- Saves results as a JSON file with an `inference` column
- Logs artifacts to MLflow

In [ ]:
# Run inference with previously fine-tuned model
response = requests.post(
    f"{BASE_URL}/pipelines/dataset-inference/run",
    json={
        # Required
        "model_name": "dataesr/my-finetuned-model",  # your fine-tuned model from step 1
        "dataset_name": "dataesr/my-training-dataset",
        "dataset_split": "eval",
        
        # Prompt config (optional)
        "prompts_config": "my-prompt-template",

        # Track model
        "tracking_config": {
            "set_active_model": "m-12345" # Model id from step 1 (when registered on mlflow)
        },
        
        # Sampling parameters (optional)
        "sampling_params": {
            "temperature": 0,
            "max_tokens": 2048,
            "seed": 42
        }
    }
)

print(response.status_code)
print(response.json())

## 3. Evaluate Model Outputs

Score the generated completions against ground truth using various metrics.

This pipeline:
- Loads the completions file generated in step 2
- Runs evaluation scorers (e.g., semantic similarity, exact match)
- Logs evaluation metrics to MLflow for comparison

In [ ]:
# Evaluate completions from inference
response = requests.post(
    f"{BASE_URL}/pipelines/dataset-evaluate/run",
    json={
        # Required
        "dataset_name": "completions_20250109_120000.json",  # output from step 2
        "model_name": "dataesr/my-finetuned-model",
        
        # Scorers to run
        "scorers": ["exact_match", "semantic_similarity"], # develop your own on core/tracking/scorers
        
        # Storage container (inference container)
        "container": "llm-completions"
    }
)

print(response.status_code)
print(response.json())

---

## Bonus: List Available Pipelines

Discover all registered pipelines and their schemas.

In [ ]:
# List all available pipelines
response = requests.get(f"{BASE_URL}/pipelines")
pipelines = response.json()

for p in pipelines:
    print(f"- {p['pipeline']}: {p['description']}")

In [ ]:
# Get schema for a specific pipeline
response = requests.get(f"{BASE_URL}/pipelines/finetune-causal")
print(response.json())